In [1]:
import cv2
import os
from datetime import datetime
import time

In [ ]:
# Captura 10 imágenes usando DroidCam over WiFi (cambia `capture_url` si hace falta)
# Opciones:
#  - DroidCam WiFi stream: http://<IP>:<PORT>/video  (ej: http://192.168.0.5:4747/video) 10.1.19.143:4747
#  - Si prefieres usar la conexión por puerto directo, prueba: "http://192.168.0.5:4747"

capture_url = 'http://192.168.26.2:4747/video'  # <--- cambia aquí si tu dispositivo usa otra ruta

carpeta = datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs(carpeta, exist_ok=True)

# Abrir stream IP en lugar de la cámara local
cap = cv2.VideoCapture(capture_url)

# Ajustes opcionales: reducir búfer para imágenes más recientes
try:
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
except Exception:
    pass

for i in range(10):  # reducido de 30 a 10 para evitar timeout
    ret, frame = cap.read()
    if ret:
        nombre_archivo = os.path.join(carpeta, f"imagen_{i+1}.jpg")
        cv2.imwrite(nombre_archivo, frame)
        print(f"Imagen {i+1}/10 guardada -> {nombre_archivo}")
        time.sleep(0.5)
    else:
        print(f"Aviso: no se recibió frame {i+1}. Verifica la URL: {capture_url}")

cap.release()
#cv2.destroyAllWindows()
print(f"Proceso completado. Imágenes guardadas en: {carpeta}")

Imagen 1/10 guardada -> 20260504_213112\imagen_1.jpg
Imagen 2/10 guardada -> 20260504_213112\imagen_2.jpg
Imagen 3/10 guardada -> 20260504_213112\imagen_3.jpg
Imagen 4/10 guardada -> 20260504_213112\imagen_4.jpg
Imagen 5/10 guardada -> 20260504_213112\imagen_5.jpg
Imagen 6/10 guardada -> 20260504_213112\imagen_6.jpg
Imagen 7/10 guardada -> 20260504_213112\imagen_7.jpg
Imagen 8/10 guardada -> 20260504_213112\imagen_8.jpg
Imagen 9/10 guardada -> 20260504_213112\imagen_9.jpg
Imagen 10/10 guardada -> 20260504_213112\imagen_10.jpg
Proceso completado. Imágenes guardadas en: 20260504_213112


In [ ]:
# Detección con YOLO (ultralytics) sobre las imágenes capturadas
# Coloca tu modelo en 'models/yolo11n.pt' o cambia la ruta below
model_path = 'models/08062026.pt'
os.makedirs('models', exist_ok=True)

try:
    from ultralytics import YOLO
    use_ultralytics = True
except Exception as e:
    print('ultralytics no está disponible en este entorno:', e)
    use_ultralytics = False

import pandas as pd
import numpy as np

def draw_box(img, xyxy, label, conf, color=(0,255,0)):
    x1, y1, x2, y2 = map(int, xyxy)
    cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
    text = f'{label} {conf:.2f}'
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
    cv2.rectangle(img, (x1, y1 - th - 6), (x1 + tw + 6, y1), color, -1)
    cv2.putText(img, text, (x1+3, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 1)

annotated_dir = os.path.join(carpeta, 'annotated')
os.makedirs(annotated_dir, exist_ok=True)

# Lista para acumular predicciones
predictions = []

image_files = [f for f in sorted(os.listdir(carpeta)) if f.lower().endswith(('.jpg','.jpeg','.png'))]
if not image_files:
    print('No se encontraron imágenes en', carpeta)
else:
    print(f'Procesando {len(image_files)} imágenes con YOLO...')

if use_ultralytics:
    try:
        model = YOLO(model_path)
        print('Modelo cargado con ultralytics:', model_path)
    except Exception as e:
        print('Error cargando el modelo ultralytics:', e)
        model = None

for fname in image_files:
    path = os.path.join(carpeta, fname)
    img = cv2.imread(path)
    if img is None:
        print('No se pudo leer', path)
        continue
    annotated = img.copy()

    detected_count = 0
    if use_ultralytics and model is not None:
        results = model(img)
        res = results[0]
        for box in getattr(res, 'boxes', []):
            # extraer coordenadas/conf/clase con fallbacks razonables
            try:
                xy = box.xyxy[0].cpu().numpy()
            except Exception:
                try:
                    xy = box.xyxy[0].numpy()
                except Exception:
                    xy = box.xyxy[0]
            try:
                conf = float(box.conf[0])
            except Exception:
                try:
                    conf = float(box.conf)
                except Exception:
                    conf = 0.0
            try:
                cls = int(box.cls[0])
            except Exception:
                try:
                    cls = int(box.cls)
                except Exception:
                    cls = 0
            label = res.names[cls] if hasattr(res, 'names') else (model.names[cls] if hasattr(model, 'names') else str(cls))

            # guardar predicción en la lista
            predictions.append({
                'filename': fname,
                'label': label,
                'confidence': conf,
                'x1': int(xy[0]),
                'y1': int(xy[1]),
                'x2': int(xy[2]),
                'y2': int(xy[3])
            })

            draw_box(annotated, xy, label, conf)
            detected_count += 1
    else:
        if not use_ultralytics:
            print('ultralytics no disponible: instala en el entorno')
        else:
            print('Modelo no cargado; omitiendo detección para', path)

    outpath = os.path.join(annotated_dir, fname)
    cv2.imwrite(outpath, annotated)
    print('Guardada anotada ->', outpath)

# ---- Guardar predicciones y generar tabla + gráficas ----
if predictions:
    df = pd.DataFrame(predictions)
    csv_path = os.path.join(carpeta, 'predictions.csv')
    df.to_csv(csv_path, index=False)
    print('Predicciones guardadas en:', csv_path)

    # Crear imagen de la tabla de predicciones
    try:
        import matplotlib.pyplot as plt
        # import seaborn as sns  # no se usa
        # from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas  # no se usa

        fig, ax = plt.subplots(figsize=(10, max(2, len(df)*0.25)))
        ax.axis('off')
        table = ax.table(cellText=df.values, colLabels=df.columns, loc='center', cellLoc='left')
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1, 1.2)
        table_path = os.path.join(carpeta, 'predictions_table.png')
        fig.tight_layout()
        fig.savefig(table_path, dpi=200, bbox_inches='tight')
        plt.close(fig)
        print('Imagen de la tabla guardada en:', table_path)
    except Exception as e:
        print('No se pudo generar la imagen de la tabla:', e)

    # Agrupar por etiqueta y calcular medias (aritmética y geométrica)
    try:
        grouped = df.groupby('label')['confidence']
        mean_df = grouped.agg(arithmetic_mean='mean', count='count')
        # geom mean: exp(mean(log(x))) but avoid zeros
        def geom_mean(x):
            arr = x.clip(lower=1e-8)
            return float(arr.prod()**(1.0/len(arr))) if len(arr)>0 else 0.0
        mean_df['geometric_mean'] = grouped.apply(geom_mean)
        mean_df = mean_df.reset_index()

        # guardar tabla de promedios
        mean_csv = os.path.join(carpeta, 'predictions_summary.csv')
        mean_df.to_csv(mean_csv, index=False)
        print('Resumen guardado en:', mean_csv)

        # graficar promedios
        fig2, ax2 = plt.subplots(figsize=(8,4))
        x = range(len(mean_df))
        ax2.bar([i-0.15 for i in x], mean_df['arithmetic_mean'], width=0.3, label='Media aritmética')
        ax2.bar([i+0.15 for i in x], mean_df['geometric_mean'], width=0.3, label='Media geométrica')
        ax2.set_xticks(x)
        ax2.set_xticklabels(mean_df['label'], rotation=45, ha='right')
        ax2.set_ylabel('Confidence')
        ax2.set_title('Comparación de medias por etiqueta')
        ax2.legend()
        fig2.tight_layout()
        summary_plot = os.path.join(carpeta, 'predictions_summary.png')
        fig2.savefig(summary_plot, dpi=200, bbox_inches='tight')
        plt.close(fig2)
        print('Gráfica de resumen guardada en:', summary_plot)
    except Exception as e:
        print('Error al generar resumen/gráfica:', e)
else:
    print('No se detectaron objetos, no se generaron tablas ni gráficas')

print('Detección finalizada. Revisa', annotated_dir)

# Gráfica de medias (aritmética y geométrica) por etiqueta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

csv_path = os.path.join(carpeta, 'predictions.csv')
if not os.path.exists(csv_path):
    print(f'No existe el archivo de predicciones: {csv_path}')
else:
    df = pd.read_csv(csv_path)
    if 'label' not in df.columns or 'confidence' not in df.columns:
        print('El archivo de predicciones no tiene las columnas esperadas.')
    else:
        grouped = df.groupby('label')['confidence']
        mean_df = grouped.agg(arithmetic_mean='mean', count='count')
        # geom mean: exp(mean(log(x))) evitando ceros
        def geom_mean(x):
            arr = np.clip(x, 1e-8, None)
            return float(np.exp(np.mean(np.log(arr)))) if len(arr)>0 else 0.0
        mean_df['geometric_mean'] = grouped.apply(geom_mean)
        mean_df = mean_df.reset_index()

        # Mostrar tabla
        display(mean_df)

        # Graficar
        fig, ax = plt.subplots(figsize=(8,4))
        x = np.arange(len(mean_df))
        ax.bar(x-0.15, mean_df['arithmetic_mean'], width=0.3, label='Media aritmética')
        ax.bar(x+0.15, mean_df['geometric_mean'], width=0.3, label='Media geométrica')
        ax.set_xticks(x)
        ax.set_xticklabels(mean_df['label'], rotation=45, ha='right')
        ax.set_ylabel('Confidence')
        ax.set_title('Comparación de medias por etiqueta')
        ax.legend()
        plt.tight_layout()
        plt.show()

Procesando 10 imágenes con YOLO...
Modelo cargado con ultralytics: models/best04052026.pt

0: 480x640 1 contenido_incorrecto, 116.8ms
Speed: 3.1ms preprocess, 116.8ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)
Guardada anotada -> 20260504_213112\annotated\imagen_1.jpg

0: 480x640 1 contenido_incorrecto, 67.5ms
Speed: 1.4ms preprocess, 67.5ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)
Guardada anotada -> 20260504_213112\annotated\imagen_10.jpg

0: 480x640 1 contenido_incorrecto, 69.9ms
Speed: 2.9ms preprocess, 69.9ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)
Guardada anotada -> 20260504_213112\annotated\imagen_2.jpg

0: 480x640 1 contenido_incorrecto, 70.0ms
Speed: 1.6ms preprocess, 70.0ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)
Guardada anotada -> 20260504_213112\annotated\imagen_3.jpg

0: 480x640 1 contenido_incorrecto, 69.8ms
Speed: 1.4ms preprocess, 69.8ms inference, 1.5ms postprocess per ima

: 

In [ ]:
# Gráfica de medias (aritmética y geométrica) por etiqueta (no interactiva)
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # backend no interactivo para evitar bloqueos
import matplotlib.pyplot as plt
import os

# Auto-detectar la carpeta más reciente
carpeta = "20260427_184721"

csv_path = os.path.join(carpeta, 'predictions.csv')
if not os.path.exists(csv_path):
    print(f'No existe el archivo de predicciones: {csv_path}')
else:
    df = pd.read_csv(csv_path)
    if 'label' not in df.columns or 'confidence' not in df.columns:
        print('El archivo de predicciones no tiene las columnas esperadas.')
    else:
        grouped = df.groupby('label')['confidence']
        mean_df = grouped.agg(arithmetic_mean='mean', count='count')
        # geom mean: exp(mean(log(x))) evitando ceros
        def geom_mean(x):
            arr = np.clip(x, 1e-8, None)
            return float(np.exp(np.mean(np.log(arr)))) if len(arr)>0 else 0.0
        mean_df['geometric_mean'] = grouped.apply(geom_mean)
        mean_df = mean_df.reset_index()

        # Mostrar la tabla pequeña (si no es muy grande)
        try:
            from IPython.display import display
            display(mean_df)
        except Exception:
            print(mean_df)

        # Graficar y guardar la figura en disco en lugar de mostrarla interactiva
        fig, ax = plt.subplots(figsize=(8,4))
        x = np.arange(len(mean_df))
        ax.bar(x-0.15, mean_df['arithmetic_mean'], width=0.3, label='Media aritmética')
        ax.bar(x+0.15, mean_df['geometric_mean'], width=0.3, label='Media geométrica')
        ax.set_xticks(x)
        ax.set_xticklabels(mean_df['label'], rotation=45, ha='right')
        ax.set_ylabel('Confidence')
        ax.set_title('Comparación de medias por etiqueta')
        ax.legend()
        plt.tight_layout()

        summary_plot = os.path.join(carpeta, 'predictions_summary.png')
        try:
            fig.savefig(summary_plot, dpi=200, bbox_inches='tight')
            print('Gráfica de resumen guardada en:', summary_plot)
        except Exception as e:
            print('Error guardando la gráfica:', e)
        finally:
            plt.close(fig)

print('Celda de gráficos completada (figuras guardadas en la carpeta).')

,label,arithmetic_mean,count,geometric_mean
0,contenido_incorrecto,0.350405,9,0.343705


: 

: 

: 

: 

: 

: 

: 